# Lab type: review
# Course: ML301 — Deep Learning with PyTorch
# Lesson: Regularisation and Generalisation
# Task: The code below is correct and working. Read each section, run it, then answer the judgment questions in the markdown cells below each block.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)
print('PyTorch version:', torch.__version__)

## Part 1: Dropout

In [ ]:
# Inverted dropout: surviving neurons are scaled by 1/(1-p) during training
# so expected activation magnitude is preserved across both modes.

torch.manual_seed(0)
x = torch.ones(1, 8)
drop = nn.Dropout(p=0.5)

# Training mode: roughly half the neurons zeroed, survivors scaled by 2.0
drop.train()
out_train = drop(x)
print('Training mode output:', out_train)
print('Non-zero values:', out_train[out_train != 0].tolist())

# Evaluation mode: all neurons pass through unchanged (no scaling, no zeroing)
drop.eval()
out_eval = drop(x)
print('\nEvaluation mode output:', out_eval)
print('Evaluation mode == all ones:', (out_eval == 1.0).all().item())

**Question 1:** During training, surviving neurons are scaled by `1 / (1 - p)`. Why is this scaling applied at training time (inverted dropout) rather than at inference time? What would happen to expected activation magnitude at inference if no scaling were applied anywhere?

*(Write your answer here.)*

<details>
<summary>🔑 Reveal answer — Q1</summary>

**Why scaling at training time (inverted dropout):** Surviving neurons are scaled by `1/(1-p)` during the forward pass so their expected value matches what inference will see — no scaling step is required at inference. This makes `model.eval()` a clean mode switch with no arithmetic change to the computation graph.

**Without any scaling:** At training time, say `p=0.4`, roughly 60% of neurons fire. At inference all neurons fire — the expected total activation is `1/0.6 ≈ 1.67×` larger than what the downstream layers were trained on. This shifts the input distribution to every subsequent layer, degrading predictions without any visible error.

</details>

**Question 2:** A model is trained for 20 epochs, then the validation loop runs without calling `model.eval()`. Dropout rate is 0.4. Describe the two problems this causes: one affecting metric reliability, one affecting reproducibility.

*(Write your answer here.)*

<details>
<summary>🔑 Reveal answer — Q2</summary>

**Problem 1 — Metric reliability:** With `p=0.4`, Dropout randomly zeroes 40% of activations on every forward pass. Validation loss and accuracy are computed on randomly masked activations, not the full model — the metric is pessimistically biased and changes between runs on identical data. You cannot reliably track whether the model is improving.

**Problem 2 — Reproducibility:** Two validation runs on the same batch produce different predictions because the Dropout mask is re-sampled each time. This makes debugging, comparison, and checkpointing unreliable — the "best epoch" depends on which random masks happened to fire during evaluation.

</details>

## Part 2: BatchNorm

In [ ]:
# BatchNorm maintains running_mean and running_var during training.
# Training mode: normalises using the current mini-batch statistics (mean, var).
# Evaluation mode: normalises using the accumulated running statistics.

torch.manual_seed(0)

# Synthetic: 100 training samples, 16 features
X_train = torch.randn(100, 16) * 3 + 5   # mean≈5, std≈3
X_val   = torch.randn(20,  16) * 3 + 5

bn = nn.BatchNorm1d(16)

# --- Training pass: accumulate running statistics ---
bn.train()
with torch.no_grad():
    for i in range(0, 100, 32):
        _ = bn(X_train[i:i+32])

print(f'running_mean (first 4): {bn.running_mean[:4].tolist()}')
print(f'running_var  (first 4): {bn.running_var[:4].tolist()}')

# --- Evaluation pass: use running statistics ---
bn.eval()
with torch.no_grad():
    out_val = bn(X_val)
print(f'\nVal output mean (approx 0): {out_val.mean().item():.4f}')
print(f'Val output std  (approx 1): {out_val.std().item():.4f}')

**Question 3:** The training loop in a colleague's script calls `model.eval()` at the start of each epoch (before both training and validation). Identify the two specific failures this causes and how each degrades training.

*(Write your answer here.)*

<details>
<summary>🔑 Reveal answer — Q3</summary>

**Failure 1 — Training loop broken:** Calling `model.eval()` before the training loop disables Dropout (no regularisation) and freezes BatchNorm to use running statistics instead of batch statistics. BatchNorm no longer adapts to the current batch distribution — the running mean/var from the previous epoch are used for normalisation, which can destabilise training or cause the statistics to stagnate.

**Failure 2 — Gradient flow affected by frozen BN:** BatchNorm in eval mode still backpropagates, but the normalisation is based on running stats that don't reflect the current batch, producing gradients that are inconsistent with what the layer actually computed. Training loss will appear erratic or fail to decrease past a certain point.

</details>

**Question 4:** During training with batch_size=4 and BatchNorm layers, training loss is noisy and validation accuracy is lower than expected. Why does a very small batch size cause BatchNorm to degrade, and what are two alternatives to consider?

*(Write your answer here.)*

<details>
<summary>🔑 Reveal answer — Q4</summary>

**Why small batches break BatchNorm:** BatchNorm estimates the mean and variance of each feature from the current mini-batch. With only 4 samples, these estimates have very high variance — the normalisation shifts wildly between batches, making training noisy and validation performance poor (validation uses running stats that were computed from these noisy estimates).

**Two alternatives:** (a) **GroupNorm** — normalises over groups of channels within each sample, independent of batch size; works well at batch size 1. (b) **LayerNorm** — normalises over all features of a single sample; common in transformers and effective when batch size is small. Alternatively, use **gradient accumulation** to simulate a larger effective batch size without increasing GPU memory.

</details>

## Part 3: AdamW and Weight Decay

In [ ]:
# AdamW decouples weight decay from the adaptive gradient update.
# Adam with weight_decay: decay is folded into the gradient before adaptive scaling,
# so larger-gradient parameters get proportionally less decay (not true L2).
# AdamW: decay applied directly to weights after the gradient step — true L2.

torch.manual_seed(0)

# Minimal MLP for demonstration
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(20, 64)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(64, 1)

    def forward(self, x):
        return self.fc2(self.relu(self.fc1(x)))

# Synthetic regression data
X = torch.randn(200, 20)
y = X[:, :5].sum(dim=1, keepdim=True) + 0.1 * torch.randn(200, 1)

dataset = TensorDataset(X, y)
loader  = DataLoader(dataset, batch_size=32, shuffle=True)
criterion = nn.MSELoss()

# --- Train with AdamW ---
model = MLP()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)

model.train()
for epoch in range(20):
    for Xb, yb in loader:
        optimizer.zero_grad()
        loss = criterion(model(Xb), yb)
        loss.backward()
        optimizer.step()

# Weight norms after training
w1_norm = model.fc1.weight.norm().item()
w2_norm = model.fc2.weight.norm().item()
print(f'fc1 weight norm (with AdamW wd=0.01): {w1_norm:.4f}')
print(f'fc2 weight norm (with AdamW wd=0.01): {w2_norm:.4f}')

model_eval = MLP()
# Reset random state for fair comparison
torch.manual_seed(0)
model_nowd = MLP()
opt_nowd   = torch.optim.AdamW(model_nowd.parameters(), lr=1e-3, weight_decay=0.0)
model_nowd.train()
for epoch in range(20):
    for Xb, yb in loader:
        opt_nowd.zero_grad()
        loss = criterion(model_nowd(Xb), yb)
        loss.backward()
        opt_nowd.step()
print(f'fc1 weight norm (no weight decay):     {model_nowd.fc1.weight.norm().item():.4f}')
print(f'fc2 weight norm (no weight decay):     {model_nowd.fc2.weight.norm().item():.4f}')

**Question 5:** `Adam(weight_decay=1e-2)` and `AdamW(weight_decay=1e-2)` are not equivalent. Describe exactly how Adam's coupling of weight decay with adaptive scaling differs from AdamW's decoupled approach. Which should you prefer for regularisation, and why?

*(Write your answer here.)*

<details>
<summary>🔑 Reveal answer — Q5</summary>

**Adam's coupled weight decay:** Adam computes an adaptive per-parameter scale factor based on gradient history. When weight decay is added to Adam's gradient before the adaptive update, the effective decay applied to each parameter is divided by the adaptive scale — parameters that receive large gradient updates (high adaptive scale) get *less* regularisation. Weight decay is not uniform across parameters.

**AdamW's decoupled weight decay:** AdamW applies weight decay directly to the parameter value *after* the adaptive gradient step, bypassing the adaptive scaling entirely. Every parameter is shrunk by the same factor `η × λ` regardless of its gradient history.

**Prefer AdamW** whenever weight decay regularisation is intentional — it behaves as L2 regularisation is designed to behave, with consistent and predictable magnitude across all parameters.

</details>

**Question 6:** Should weight decay be applied to bias terms and BatchNorm parameters? What is the conventional practice, and what is the practical effect on model behaviour if you regularise these parameters?

*(Write your answer here.)*

<details>
<summary>🔑 Reveal answer — Q6</summary>

**Conventional practice:** Exclude bias terms and all BatchNorm parameters (both `weight` and `bias`) from weight decay. In PyTorch this is done by creating two parameter groups — one with `weight_decay` applied (weight matrices only) and one without (biases and BN params).

**Why:** Bias terms are scalars that shift layer outputs; shrinking them toward zero adds noise without improving generalisation. BatchNorm's `weight` (γ) and `bias` (β) are scale-and-shift parameters controlling normalised output — regularising them distorts the learned normalisation distribution in ways that don't correspond to reducing model complexity. The practical effect of regularising them is degraded performance on tasks where the BN statistics matter (most tasks).

</details>

## Part 4: Layer Ordering — Linear → BatchNorm → ReLU

In [ ]:
# Standard placement: Linear → BatchNorm → Activation
# BatchNorm normalises the pre-activation distribution, giving ReLU a consistent
# zero-centred input. This keeps gradients well-scaled and avoids dead neurons.

torch.manual_seed(0)
x = torch.randn(32, 20)

# Correct ordering
correct_block = nn.Sequential(
    nn.Linear(20, 64),
    nn.BatchNorm1d(64),
    nn.ReLU(),
)

# Incorrect: BatchNorm applied after ReLU — normalises already-non-negative values,
# losing the centring benefit and partially defeating the purpose of normalisation.
incorrect_block = nn.Sequential(
    nn.Linear(20, 64),
    nn.ReLU(),
    nn.BatchNorm1d(64),
)

out_correct   = correct_block(x)
out_incorrect = incorrect_block(x)

print('Correct   (BN before ReLU) — mean: {:.4f}, std: {:.4f}'.format(
    out_correct.mean().item(), out_correct.std().item()))
print('Incorrect (BN after ReLU) — mean: {:.4f}, std: {:.4f}'.format(
    out_incorrect.mean().item(), out_incorrect.std().item()))
print()
# BN after ReLU normalises non-negative values: mean shifts upward
# confirming BN now operates on a truncated (non-centred) distribution
print('Fraction of zeros in incorrect output:', (out_incorrect == 0).float().mean().item())

**Question 7:** Some research papers place BatchNorm *after* the activation. What is the practical risk of that ordering versus the standard Linear → BatchNorm → ReLU? When might post-activation normalisation still be acceptable?

*(Write your answer here.)*

<details>
<summary>🔑 Reveal answer — Q7</summary>

**Risk of post-activation BatchNorm (Linear → ReLU → BatchNorm):** ReLU zeroes all negative activations, producing a half-rectified distribution. BatchNorm normalises this asymmetric distribution, which is less effective than normalising pre-activation values (which are approximately Gaussian). Additionally, the sparsity pattern (which neurons were zero) is information that is lost before normalisation — the next layer sees a normalised signal but not the original activation structure.

**When it is still acceptable:** In residual networks (original ResNet paper), the shortcut connection bypasses the activation and preserves variance through the skip path. Post-activation BN in the residual branch does less damage because the shortcut provides an unmodified signal. Many modern architectures (Pre-Activation ResNet, Transformers) have moved to pre-norm (norm before activation) precisely to avoid this issue.

</details>

## Summary

> **Final check:** Answer in one sentence each.

1. What does inverted dropout scaling achieve, and when is the scale factor applied?
2. Which two BatchNorm failure modes result from forgetting to switch between train and eval mode?
3. In one sentence, why should you prefer AdamW over Adam when weight decay regularisation matters?
4. What is the correct layer ordering for a Linear block with BatchNorm and ReLU, and why?

<details>
<summary>🔑 Reveal summary answers</summary>

1. **Inverted dropout scaling:** Surviving activations are scaled by `1/(1-p)` at training time, so their expected value is unchanged at inference — `model.eval()` requires no compensating scale factor.

2. **Two BatchNorm failures from wrong mode:** In training mode during validation, BatchNorm uses noisy batch statistics instead of running stats (wrong normalisation); in eval mode during training, BatchNorm uses frozen running stats instead of batch stats (statistics stop updating, degrading the normalisation).

3. **Prefer AdamW over Adam with weight decay:** AdamW decouples weight decay from the adaptive gradient scaling so every parameter receives consistent regularisation; Adam's coupled formulation causes weight decay magnitude to vary inversely with gradient history, making it weaker for high-gradient parameters.

4. **Correct ordering — Linear → BatchNorm → ReLU:** Normalise before the activation so BatchNorm operates on a near-Gaussian distribution; applying ReLU first produces a half-rectified input whose sparsity pattern BatchNorm cannot effectively normalise.

</details>